# LeetCode 648 — Replace Words bằng Python

Link đề: https://leetcode.com/problems/replace-words/ · Độ khó: **Medium** · Chủ đề: Trie, Array, Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def replaceWords(self, dictionary: List[str], sentence: str) -> str:
        ...
```

Ghi chú: bài cho một danh sách **gốc từ (root)** và một câu; mỗi từ trong câu nếu bắt đầu bằng một gốc thì được thay bằng gốc **ngắn nhất** như vậy. Notebook đi từ vét cạn `startswith`, qua tập băm các tiền tố, tới **Trie** (cây tiền tố) và cách **sắp xếp + tìm kiếm nhị phân**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: thay mỗi từ trong câu bằng **tiền tố ngắn nhất** của nó có trong từ điển; từ nào không có tiền tố trong từ điển thì giữ nguyên.
2. Nắm các công cụ Python: `str.split()`, `" ".join(...)`, `str.startswith`, cắt tiền tố `word[:k]`, `dict`/`setdefault`, `set` tra cứu `O(1)`, `sorted` + `bisect`, định nghĩa `class` với `__init__`.
3. Đi từ **thử từng root với từng từ** → **kiểm tra từng tiền tố trong `set`** → **Trie** → **sắp xếp roots + bisect**.
4. Hiểu mấu chốt: **Trie** dùng chung các tiền tố, cho phép đi dọc theo từ và **dừng ngay ở root đầu tiên gặp** (đó chính là root ngắn nhất).
5. Tránh các lỗi kinh điển: lấy root khớp **đầu tiên trong từ điển** thay vì **ngắn nhất**, đi tiếp quá `is_end` (lấy root dài nhất), dùng `root in word` thay cho kiểm tra tiền tố, và trả về tiền tố "đi dở" khi chưa gặp root.

## 2. Đề bài & ví dụ

Trong tiếng Anh có khái niệm **root (gốc)**: khi thêm một số chữ sau root ta được một **derivative (từ phái sinh)**, ví dụ root `"help"` + `"ful"` = `"helpful"`.

Cho `dictionary` gồm nhiều root và `sentence` gồm các từ cách nhau bằng dấu cách. Hãy thay **mỗi từ phái sinh** trong câu bằng root tạo ra nó. Nếu một từ có thể thay bằng nhiều root, hãy dùng root **ngắn nhất**. Trả về câu sau khi thay.

**Ví dụ 1**

- Input: `dictionary = ["cat","bat","rat"]`, `sentence = "the cattle was rattled by the battery"`
- Output: `"the cat was rat by the bat"`

**Ví dụ 2**

- Input: `dictionary = ["a","b","c"]`, `sentence = "aadsfasf absbs bbab cadsfafs"`
- Output: `"a a b c"`

**Ràng buộc:**

- `1 <= dictionary.length <= 1000`
- `1 <= dictionary[i].length <= 100`, `dictionary[i]` chỉ gồm chữ thường
- `1 <= sentence.length <= 10^6`, `sentence` chỉ gồm chữ thường và dấu cách
- Số từ trong câu từ `1` đến `1000`; mỗi từ dài từ `1` đến `1000`
- Hai từ liên tiếp cách nhau đúng một dấu cách; không có dấu cách ở đầu/cuối câu

Chú ý: "derivative" ở đây hiểu đơn giản là từ **bắt đầu bằng** root (root là **tiền tố**). Một từ bằng đúng root cũng được "thay" bằng chính nó.

In [1]:
from typing import List
from bisect import bisect_right
import random

examples = [
    (["cat", "bat", "rat"], "the cattle was rattled by the battery", "the cat was rat by the bat"),
    (["a", "b", "c"], "aadsfasf absbs bbab cadsfafs", "a a b c"),
]
for dictionary, sentence, expected in examples:
    print(dictionary, "|", sentence, "->", expected)

['cat', 'bat', 'rat'] | the cattle was rattled by the battery -> the cat was rat by the bat
['a', 'b', 'c'] | aadsfasf absbs bbab cadsfafs -> a a b c


## 3. Kiến thức nền

### 3.1. `split()` và `join()`

`sentence.split()` cắt chuỗi thành **list các từ** theo khoảng trắng; `" ".join(words)` nối list lại bằng đúng một dấu cách. Vì đề đảm bảo các từ cách nhau đúng một dấu cách, `" ".join(sentence.split())` luôn trả lại chính câu gốc.

In [2]:
sentence = "the cattle was rattled"
words = sentence.split()
print(words)
print(" ".join(words))
print(" ".join(words) == sentence)
print(len(words), "từ")

['the', 'cattle', 'was', 'rattled']
the cattle was rattled
True
4 từ


### 3.2. Tiền tố: `startswith` và cắt `word[:k]`

**Tiền tố (prefix)** của một chuỗi là đoạn **bắt đầu từ ký tự đầu**. `word.startswith(root)` trả `True` khi `root` là tiền tố của `word`. `word[:k]` lấy `k` ký tự đầu (tiền tố độ dài `k`). Một chuỗi luôn là tiền tố của chính nó.

Phân biệt: `root in word` kiểm tra `root` là **chuỗi con ở đâu đó** (không nhất thiết ở đầu) — đây là lỗi hay gặp ở mục 13.

In [3]:
word = "cattle"
print(word.startswith("cat"), word.startswith("tle"), word.startswith("cattle"))
print([word[:k] for k in range(1, len(word) + 1)])
print("tle" in word, "(chuỗi con, nhưng KHÔNG phải tiền tố)")

True False True
['c', 'ca', 'cat', 'catt', 'cattl', 'cattle']
True (chuỗi con, nhưng KHÔNG phải tiền tố)


### 3.3. `dict`, `setdefault` và `in`

`dict` lưu cặp khóa-giá trị. `d.setdefault(k, v)` trả `d[k]` nếu `k` đã có, ngược lại **tạo** `d[k] = v` rồi trả `v`. `k in d` kiểm tra khóa tồn tại, trung bình `O(1)`. Ta sẽ dùng dict lồng dict để mô tả cây.

In [4]:
d = {}
d.setdefault("c", {})["a"] = 1
print(d)
node = d.setdefault("c", {})       # đã có -> trả về dict cũ, không tạo mới
print(node, "| 'a' in node:", "a" in node, "| 'x' in node:", "x" in node)

{'c': {'a': 1}}
{'a': 1} | 'a' in node: True | 'x' in node: False


### 3.4. Trie (cây tiền tố) là gì?

**Trie** là cây mà mỗi **cạnh** là một ký tự; đường đi từ gốc xuống một nút ghép lại thành một tiền tố. Các từ chung tiền tố dùng chung đường đi. Mỗi nút có thêm cờ `is_end` cho biết "đường đi tới đây là **một từ hoàn chỉnh** (một root)".

Ví dụ trie của `["cat", "car", "bat"]`:

```
(gốc)
 ├─ c ─ a ─ t*      (* = is_end)
 │       └─ r*
 └─ b ─ a ─ t*
```

Ta cài nút bằng class với `children` (dict ký tự → nút con) và `is_end`.

In [5]:
class TrieNode:
    def __init__(self):
        self.children = {}      # ký tự -> TrieNode
        self.is_end = False     # đường đi tới đây có phải một root hoàn chỉnh?

def insert(root, word):
    node = root
    for ch in word:
        if ch not in node.children:
            node.children[ch] = TrieNode()
        node = node.children[ch]
    node.is_end = True

def dump(node, prefix=""):
    """In trie ra dạng liệt kê các từ được lưu."""
    out = []
    if node.is_end:
        out.append(prefix)
    for ch in sorted(node.children):
        out += dump(node.children[ch], prefix + ch)
    return out

demo = TrieNode()
for w in ["cat", "car", "bat"]:
    insert(demo, w)
print(dump(demo))
print("nút con của gốc:", sorted(demo.children))
print("nhánh c ->", sorted(demo.children["c"].children), "-> a ->", sorted(demo.children["c"].children["a"].children))

['bat', 'car', 'cat']
nút con của gốc: ['b', 'c']
nhánh c -> ['a'] -> a -> ['r', 't']


### 3.5. `set` và tra cứu `O(1)`

`set` là tập hợp không trùng, kiểm tra `x in s` trung bình `O(1)` (băm). Ta sẽ dùng `set` các root để hỏi "tiền tố này có phải root không?" nhanh.

In [6]:
roots = {"cat", "bat", "rat"}
print("cat" in roots, "cattle" in roots, "ca" in roots)
print(len(roots), set(["a", "a", "b"]))

True False False
3 {'b', 'a'}


### 3.6. `sorted` và `bisect_right`

`bisect_right(arr, x)` trên list **đã sắp xếp** trả về vị trí chèn `x` sau mọi phần tử `<= x` (tìm nhị phân, `O(log n)`). Khi đó `arr[idx - 1]` là **phần tử lớn nhất `<= x`** (người tiền nhiệm). Ta sẽ dùng nó ở Cách 4.

In [7]:
arr = sorted(["bat", "cat", "rat", "car"])
print(arr)
for x in ["cattle", "car", "dog", "apple"]:
    idx = bisect_right(arr, x)
    print(f"{x!r}: idx = {idx}, người tiền nhiệm = {arr[idx - 1] if idx else None}")

['bat', 'car', 'cat', 'rat']
'cattle': idx = 3, người tiền nhiệm = cat
'car': idx = 2, người tiền nhiệm = car
'dog': idx = 3, người tiền nhiệm = cat
'apple': idx = 0, người tiền nhiệm = None


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 với roots `cat, bat, rat`:

- `the`: chữ đầu `t` không phải đầu của root nào → giữ `the`.
- `cattle`: bắt đầu `c-a-t` — khớp root `cat` → thay bằng `cat`.
- `was`: bắt đầu `w` → giữ.
- `rattled`: `r-a-t` → `rat`.
- `by`: bắt đầu `b` nhưng `by` không khớp `bat` (`y != a`) → giữ `by`.
- `battery`: `b-a-t` → `bat`.

Câu kết quả: `the cat was rat by the bat`.

Vấn đề "ngắn nhất": nếu root có `["cat", "c"]` thì từ `cattle` có hai tiền tố là root (`c` và `cat`); ta phải chọn `c`. Cách tự nhiên: **thử các tiền tố từ ngắn tới dài** (`c`, `ca`, `cat`, ...) và dừng ở tiền tố đầu tiên là root — nó chắc chắn là root ngắn nhất. Đây là ý nền tảng của mọi cách sau.

In [8]:
def shortest_prefix_root(word, roots):
    for k in range(1, len(word) + 1):          # từ tiền tố ngắn nhất
        if word[:k] in roots:
            return word[:k]
    return word

for w in ["cattle", "rattled", "by", "the"]:
    print(f"{w!r:10} -> {shortest_prefix_root(w, {'cat', 'bat', 'rat'})!r}")
print("hai root lồng nhau:", shortest_prefix_root("cattle", {"cat", "c"}))

'cattle'   -> 'cat'
'rattled'  -> 'rat'
'by'       -> 'by'
'the'      -> 'the'
hai root lồng nhau: c


## 5. Cách 1 — Brute Force: thử từng root với từng từ

### Ý tưởng

Với mỗi từ `w` của câu, duyệt **mọi root** `r` trong từ điển; nếu `w.startswith(r)` và `r` **ngắn hơn** kết quả hiện tại thì cập nhật. Cuối cùng nối các từ đã xử lý.

### Code

In [9]:
def replace_brute(dictionary, sentence):
    out = []
    for w in sentence.split():
        best = w
        for r in dictionary:
            if w.startswith(r) and len(r) < len(best):
                best = r
        out.append(best)
    return " ".join(out)

for dictionary, sentence, expected in examples:
    got = replace_brute(dictionary, sentence)
    print(repr(got))
    assert got == expected
print(replace_brute(["cat", "c"], "cattle"))       # root lồng nhau: phải ra 'c'

'the cat was rat by the bat'
'a a b c'
c


### Độ phức tạp và nút thắt

- Gọi `W` là số từ, `n` số root, `m` độ dài root tối đa. Mỗi cặp (từ, root) tốn tới `O(m)` cho `startswith`.
- Time `O(W * n * m)`: với `W = 1000`, `n = 1000`, `m = 100` là `10^8` ký tự — chậm trong Python.
- Space `O(L)` cho kết quả (`L` là độ dài câu).

**Nút thắt:** mỗi từ phải so với **tất cả** root dù hầu hết root hoàn toàn không liên quan (khác ngay chữ cái đầu). Ta đang **lặp lại** việc so các tiền tố giống nhau (ví dụ `cat` và `car` cùng bắt đầu `ca`). Đếm thử số phép so sánh `startswith`:

In [10]:
def count_startswith_calls(dictionary, sentence):
    return len(sentence.split()) * len(dictionary)

rng = random.Random(1)
roots = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 5))) for _ in range(1000)]
sent = " ".join("".join(rng.choice("abc") for _ in range(rng.randint(1, 8))) for _ in range(1000))
print("số lần gọi startswith:", count_startswith_calls(roots, sent))

số lần gọi startswith: 1000000


## 6. Ý tưởng thuật toán / bước đột phá

Ở mục 4 ta thấy: duyệt tiền tố **ngắn → dài** của từ và dừng ở tiền tố đầu tiên thuộc từ điển. Việc cần làm nhanh là **"tiền tố hiện tại có phải root không, và nó còn có thể mở rộng thành root dài hơn không?"** — chính là việc của **Trie**:

1. **Xây Trie** từ mọi root (đánh dấu `is_end` ở nút cuối mỗi root).
2. Với mỗi từ, **đi từ gốc Trie theo từng ký tự** của từ:
   - Nếu ký tự không có nút con tương ứng → **không root nào là tiền tố của từ** (đường đã đứt) → giữ nguyên từ.
   - Nếu tới nút có `is_end = True` → vừa đi qua một root hoàn chỉnh. Vì ta đi **từ ngắn tới dài**, đây là root **ngắn nhất** → trả `word[:k]` ngay, không đi tiếp.
   - Nếu hết từ mà chưa gặp `is_end` → giữ nguyên từ (từ ngắn hơn mọi root cùng đường đi).
3. Nối lại các từ.

Chi phí mỗi từ chỉ `O(min(độ dài từ, độ dài root tối đa))`, **không phụ thuộc số root**.

In [11]:
# Quan sát: đi dọc Trie theo từ "cattle" với roots cat, car, bat
t = TrieNode()
for w in ["cat", "car", "bat"]:
    insert(t, w)

node = t
for k, ch in enumerate("cattle", start=1):
    if ch not in node.children:
        print(f"ký tự {ch!r} (thứ {k}): không có nút con -> đường đứt")
        break
    node = node.children[ch]
    print(f"ký tự {ch!r} (thứ {k}): có nút con, is_end = {node.is_end}", "-> DỪNG, root =", repr("cattle"[:k]) if node.is_end else "")
    if node.is_end:
        break

ký tự 'c' (thứ 1): có nút con, is_end = False -> DỪNG, root = 
ký tự 'a' (thứ 2): có nút con, is_end = False -> DỪNG, root = 
ký tự 't' (thứ 3): có nút con, is_end = True -> DỪNG, root = 'cat'


## 7. Các cách tiếp theo

Phần này có thêm 3 cách (tổng 4 cách): **tập băm các tiền tố**, **Trie** (lời giải nộp), **sắp xếp + `bisect`**.

### 7.1. Cách 2 — Tập băm (`set`) các root, thử từng tiền tố của từ

**Giải thích từng đoạn.**

1. `roots = set(dictionary)` và `max_len = max(len(r) for r in roots)`: độ dài root tối đa.
2. Với mỗi từ `w`, thử `k = 1, 2, ..., min(len(w), max_len)`: nếu `w[:k] in roots` thì thay `w` bằng `w[:k]` và dừng (đây là tiền tố ngắn nhất).
3. Giới hạn `k <= max_len` quan trọng: không có root nào dài hơn `max_len` nên thử tiếp là vô ích.

In [12]:
def replace_prefix_set(dictionary, sentence):
    roots = set(dictionary)
    max_len = max(len(r) for r in roots)
    out = []
    for w in sentence.split():
        for k in range(1, min(len(w), max_len) + 1):
            if w[:k] in roots:
                w = w[:k]
                break
        out.append(w)
    return " ".join(out)

for dictionary, sentence, expected in examples:
    got = replace_prefix_set(dictionary, sentence)
    print(repr(got))
    assert got == expected

'the cat was rat by the bat'
'a a b c'


In [13]:
class Solution:
    def replaceWords(self, dictionary: List[str], sentence: str) -> str:
        roots = set(dictionary)
        max_len = max(len(r) for r in roots)
        out = []
        for w in sentence.split():
            for k in range(1, min(len(w), max_len) + 1):
                if w[:k] in roots:
                    w = w[:k]
                    break
            out.append(w)
        return " ".join(out)

print(Solution().replaceWords(["cat", "bat", "rat"], "the cattle was rattled by the battery"))

the cat was rat by the bat


**Độ phức tạp:** mỗi từ thử tối đa `m` tiền tố, mỗi lần cắt + băm tốn `O(k)` → Time `O(W * m^2)` (với `W = 1000`, `m = 100`: `10^7`), Space `O(n * m)` cho tập root. Gọn, nhanh với ràng buộc này, nhưng cắt chuỗi lặp lại nên Trie (không cắt) hiệu quả hơn.

### 7.2. Cách 3 — Trie (lời giải nộp)

**Giải thích từng đoạn.**

1. Lớp `Trie` giữ `children` (dict) và `is_end`. Phương thức `insert(word)` đi/tạo nút theo từng ký tự rồi đặt `is_end = True`.
2. Phương thức `shortest_root(word)`: đi theo từng ký tự `ch` (kèm chỉ số `k`). Nếu `ch` không có trong `children` → `return word`. Nếu có, chuyển xuống nút con; nếu nút đó `is_end` → `return word[:k + 1]`. Hết vòng lặp mà chưa dừng → `return word`.
3. `replaceWords`: xây Trie từ `dictionary`, rồi `" ".join(trie.shortest_root(w) for w in sentence.split())`.

In [14]:
class Trie:
    def __init__(self):
        self.children = {}
        self.is_end = False

    def insert(self, word):
        node = self
        for ch in word:
            if ch not in node.children:
                node.children[ch] = Trie()
            node = node.children[ch]
        node.is_end = True

    def shortest_root(self, word):
        node = self
        for k, ch in enumerate(word):
            if ch not in node.children:
                return word                 # đường đứt: không có root nào là tiền tố
            node = node.children[ch]
            if node.is_end:
                return word[:k + 1]         # root hoàn chỉnh đầu tiên = ngắn nhất
        return word                         # hết từ mà chưa gặp root

def replace_trie(dictionary, sentence):
    trie = Trie()
    for root in dictionary:
        trie.insert(root)
    return " ".join(trie.shortest_root(w) for w in sentence.split())

for dictionary, sentence, expected in examples:
    got = replace_trie(dictionary, sentence)
    print(repr(got))
    assert got == expected

'the cat was rat by the bat'
'a a b c'


In [15]:
class Trie:
    def __init__(self):
        self.children = {}
        self.is_end = False

    def insert(self, word):
        node = self
        for ch in word:
            if ch not in node.children:
                node.children[ch] = Trie()
            node = node.children[ch]
        node.is_end = True

    def shortest_root(self, word):
        node = self
        for k, ch in enumerate(word):
            if ch not in node.children:
                return word
            node = node.children[ch]
            if node.is_end:
                return word[:k + 1]
        return word


class Solution:
    def replaceWords(self, dictionary: List[str], sentence: str) -> str:
        trie = Trie()
        for root in dictionary:
            trie.insert(root)
        return " ".join(trie.shortest_root(w) for w in sentence.split())

print(Solution().replaceWords(["a", "b", "c"], "aadsfasf absbs bbab cadsfafs"))

a a b c


**Độ phức tạp:** xây Trie `O(tổng độ dài root)` (tối đa `10^5`); mỗi từ đi tối đa `min(|w|, m)` bước → Time tổng `O(tổng root + L)` với `L` là tổng độ dài các từ; Space `O(tổng độ dài root)` (tối đa `10^5` nút).

### 7.3. Cách 4 — Sắp xếp roots + `bisect` (tìm người tiền nhiệm)

**Ý tưởng.** Bước 1: **lọc root "tối giản"**: sắp xếp root theo thứ tự từ điển, bỏ root nào có tiền tố là một root đã giữ. Sau lọc, **không root nào là tiền tố của root khác**. Bước 2: với từ `w`, nếu có root `r` là tiền tố của `w`, thì `r` chính là **người tiền nhiệm** của `w` trong danh sách đã sắp xếp (root lớn nhất `<= w`). Lý do: mọi chuỗi nằm giữa `r` và `w` theo thứ tự từ điển (khi `r` là tiền tố của `w`) đều có tiền tố `r`; nếu có root như vậy khác `r` thì nó có `r` làm tiền tố — mâu thuẫn tính "không tiền tố lẫn nhau". Nên chỉ cần `bisect_right` rồi kiểm tra `w.startswith(roots[idx - 1])`.

Điều này cũng cho **root ngắn nhất** vì sau lọc chỉ còn root không chứa root khác làm tiền tố (mọi root dài hơn bị loại).

In [16]:
def minimal_roots(dictionary):
    """Sắp xếp theo từ điển rồi bỏ root có tiền tố là root đã giữ."""
    kept = []
    for r in sorted(set(dictionary)):
        if kept and r.startswith(kept[-1]):
            continue
        kept.append(r)
    return kept

print(minimal_roots(["cat", "c", "cattle", "bat", "ba", "rat"]))

def replace_bisect(dictionary, sentence):
    roots = minimal_roots(dictionary)
    out = []
    for w in sentence.split():
        idx = bisect_right(roots, w)
        if idx and w.startswith(roots[idx - 1]):
            w = roots[idx - 1]
        out.append(w)
    return " ".join(out)

for dictionary, sentence, expected in examples:
    got = replace_bisect(dictionary, sentence)
    print(repr(got))
    assert got == expected
print(replace_bisect(["cat", "c"], "cattle"))

['ba', 'c', 'rat']
'the cat was rat by the bat'
'a a b c'
c


In [17]:
class Solution:
    def replaceWords(self, dictionary: List[str], sentence: str) -> str:
        kept = []
        for r in sorted(set(dictionary)):
            if kept and r.startswith(kept[-1]):
                continue
            kept.append(r)
        out = []
        for w in sentence.split():
            idx = bisect_right(kept, w)
            if idx and w.startswith(kept[idx - 1]):
                w = kept[idx - 1]
            out.append(w)
        return " ".join(out)

print(Solution().replaceWords(["cat", "bat", "rat"], "the cattle was rattled by the battery"))

the cat was rat by the bat


**Độ phức tạp:** sắp xếp `O(n * m * log n)` (so sánh chuỗi tốn `O(m)`), mỗi từ `O(m * log n)` (bisect + `startswith`) → Time `O(n m log n + W m log n)`, Space `O(n * m)`. Ý tưởng độc đáo nhưng khó chứng minh hơn Trie; chủ yếu để mở rộng tư duy "sắp xếp + người tiền nhiệm".

## 8. TRACE từng bước

### 8.1. Trace xây Trie với `["cat", "bat", "rat"]`

| root | đường đi | nút tạo mới | `is_end` đặt ở |
|---|---|---|---|
| `cat` | gốc → c → a → t | `c`, `a`, `t` (3 nút) | `t` |
| `bat` | gốc → b → a → t | `b`, `a`, `t` (3 nút) | `t` |
| `rat` | gốc → r → a → t | `r`, `a`, `t` (3 nút) | `t` |

Tổng 9 nút (không tính gốc) vì ba root không chung tiền tố nào ngoài gốc: các nút `a` (và `t`) ở ba nhánh là **khác nhau**.

In [18]:
def trace_build(dictionary):
    root = Trie()
    total = 0
    for word in dictionary:
        node = root
        created = []
        for ch in word:
            if ch not in node.children:
                node.children[ch] = Trie()
                created.append(ch)
                total += 1
            node = node.children[ch]
        node.is_end = True
        print(f"chèn {word!r}: tạo mới {created if created else 'không nút nào'}; is_end ở ký tự cuối {word[-1]!r}")
    print("tổng số nút (không tính gốc):", total)
    return root

trie = trace_build(["cat", "bat", "rat"])
trie2 = trace_build(["cat", "car", "c", "cattle"])

chèn 'cat': tạo mới ['c', 'a', 't']; is_end ở ký tự cuối 't'
chèn 'bat': tạo mới ['b', 'a', 't']; is_end ở ký tự cuối 't'
chèn 'rat': tạo mới ['r', 'a', 't']; is_end ở ký tự cuối 't'
tổng số nút (không tính gốc): 9
chèn 'cat': tạo mới ['c', 'a', 't']; is_end ở ký tự cuối 't'
chèn 'car': tạo mới ['r']; is_end ở ký tự cuối 'r'
chèn 'c': tạo mới không nút nào; is_end ở ký tự cuối 'c'
chèn 'cattle': tạo mới ['t', 'l', 'e']; is_end ở ký tự cuối 'e'
tổng số nút (không tính gốc): 7


Quan sát trie thứ hai: `cat`, `car`, `c`, `cattle` **chung** nhánh `c`, và `cat` chung với `cattle`, nên nút mới rất ít. Cũng thấy `c` bị đánh dấu `is_end` dù nằm giữa đường đi tới `cat`.

### 8.2. Trace tìm root cho từng từ (ví dụ 1)

| từ | đi theo ký tự | kết quả |
|---|---|---|
| `the` | `t` không có nút con ở gốc | đường đứt → `the` |
| `cattle` | `c` (chưa end), `a` (chưa end), `t` (`is_end`) | dừng ở `k = 2` → `cat` |
| `was` | `w` không có | `was` |
| `rattled` | `r`, `a`, `t` (`is_end`) | `rat` |
| `by` | `b` có; `y` không có con của `b` (chỉ có `a`) | đường đứt → `by` |
| `the` | giống trên | `the` |
| `battery` | `b`, `a`, `t` (`is_end`) | `bat` |

In [19]:
def trace_replace(dictionary, sentence):
    trie = Trie()
    for r in dictionary:
        trie.insert(r)
    result = []
    for w in sentence.split():
        node = trie
        answer = None
        steps = []
        for k, ch in enumerate(w):
            if ch not in node.children:
                steps.append(f"{ch!r}: không có -> đứt")
                answer = w
                break
            node = node.children[ch]
            steps.append(f"{ch!r}: có (is_end={node.is_end})")
            if node.is_end:
                answer = w[:k + 1]
                break
        if answer is None:
            answer = w
            steps.append("hết từ mà chưa gặp root")
        print(f"{w!r:10} | " + "; ".join(steps) + f"  => {answer!r}")
        result.append(answer)
    return " ".join(result)

res = trace_replace(["cat", "bat", "rat"], "the cattle was rattled by the battery")
print(res)
assert res == "the cat was rat by the bat"

'the'      | 't': không có -> đứt  => 'the'
'cattle'   | 'c': có (is_end=False); 'a': có (is_end=False); 't': có (is_end=True)  => 'cat'
'was'      | 'w': không có -> đứt  => 'was'
'rattled'  | 'r': có (is_end=False); 'a': có (is_end=False); 't': có (is_end=True)  => 'rat'
'by'       | 'b': có (is_end=False); 'y': không có -> đứt  => 'by'
'the'      | 't': không có -> đứt  => 'the'
'battery'  | 'b': có (is_end=False); 'a': có (is_end=False); 't': có (is_end=True)  => 'bat'
the cat was rat by the bat


In [20]:
# Trace ca khó: root lồng nhau (phải chọn root NGẮN nhất) và root dài hơn từ
res = trace_replace(["cat", "c", "cattle", "ca"], "cattle catalog cow c ca")
print(res)
assert res == "c c c c c"
res = trace_replace(["hello"], "hell hello helloworld")
print(res)
assert res == "hell hello hello"

'cattle'   | 'c': có (is_end=True)  => 'c'
'catalog'  | 'c': có (is_end=True)  => 'c'
'cow'      | 'c': có (is_end=True)  => 'c'
'c'        | 'c': có (is_end=True)  => 'c'
'ca'       | 'c': có (is_end=True)  => 'c'
c c c c c
'hell'     | 'h': có (is_end=False); 'e': có (is_end=False); 'l': có (is_end=False); 'l': có (is_end=False); hết từ mà chưa gặp root  => 'hell'
'hello'    | 'h': có (is_end=False); 'e': có (is_end=False); 'l': có (is_end=False); 'l': có (is_end=False); 'o': có (is_end=True)  => 'hello'
'helloworld' | 'h': có (is_end=False); 'e': có (is_end=False); 'l': có (is_end=False); 'l': có (is_end=False); 'o': có (is_end=True)  => 'hello'
hell hello hello


## 9. Code nộp LeetCode

Lời giải nộp là **Trie** (Cách 3). Trên LeetCode, nên đặt lớp phụ `Trie` ngay phía trên `Solution` trong cùng file.

In [21]:
class Trie:
    def __init__(self):
        self.children = {}
        self.is_end = False

    def insert(self, word: str) -> None:
        node = self
        for ch in word:
            if ch not in node.children:
                node.children[ch] = Trie()
            node = node.children[ch]
        node.is_end = True

    def shortest_root(self, word: str) -> str:
        node = self
        for k, ch in enumerate(word):
            if ch not in node.children:
                return word
            node = node.children[ch]
            if node.is_end:
                return word[:k + 1]
        return word


class Solution:
    def replaceWords(self, dictionary: List[str], sentence: str) -> str:
        trie = Trie()
        for root in dictionary:
            trie.insert(root)
        return " ".join(trie.shortest_root(w) for w in sentence.split())


sol = Solution()
for dictionary, sentence, expected in examples:
    got = sol.replaceWords(dictionary, sentence)
    print(repr(got))
    assert got == expected

'the cat was rat by the bat'
'a a b c'


## 10. Vì sao đúng

**Mệnh đề 1 (Trie lưu đúng tập tiền tố).** Với một xâu `p`, đường đi từ gốc theo `p` tồn tại khi và chỉ khi `p` là **tiền tố của ít nhất một root**. Nút cuối của đường đi có `is_end = True` khi và chỉ khi `p` **chính là một root**. (Do `insert` tạo đúng các nút dọc theo từng root và chỉ đặt `is_end` ở nút cuối.)

**Mệnh đề 2 (dừng đúng chỗ).** Khi duyệt từ `w` theo từng ký tự, sau `k + 1` bước ta ở nút ứng với `w[:k+1]`.

- Nếu ở bước nào đó ký tự không có nút con: `w[:k+1]` không là tiền tố của root nào, nên **không root nào dài hơn `k` là tiền tố của `w`**; đồng thời ở các bước trước đó không gặp `is_end` nên **không có root nào ngắn hơn là tiền tố của `w`** → từ giữ nguyên. Đúng.
- Nếu gặp `is_end` lần đầu ở bước `k + 1`: `w[:k+1]` là một root và mọi tiền tố ngắn hơn **không** là root (chưa từng gặp `is_end`), do đó nó là root **ngắn nhất** là tiền tố của `w`. Trả `w[:k+1]` là đúng.
- Hết từ mà chưa gặp `is_end`: không tiền tố nào của `w` (kể cả chính `w`) là root → giữ nguyên.

**Các cách khác:** Cách 2 thử tiền tố từ ngắn tới dài nên cũng dừng ở root ngắn nhất; Cách 1 chọn min theo độ dài trên mọi root khớp; Cách 4 dùng tính chất người tiền nhiệm đã chứng minh ở mục 7.3. Ta kiểm tra tính chất người tiền nhiệm bằng code.

In [22]:
# Kiểm chứng thực nghiệm: sau khi lọc root tối giản, root là tiền tố của w <=> nó là người tiền nhiệm của w
rng = random.Random(10)
checked = 0
for _ in range(2000):
    dictionary = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 6))]
    kept = minimal_roots(dictionary)
    # tính chất: không root nào là tiền tố của root khác
    for x in kept:
        for y in kept:
            assert x == y or not y.startswith(x)
    w = "".join(rng.choice("ab") for _ in range(rng.randint(1, 8)))
    prefixes = [r for r in kept if w.startswith(r)]
    assert len(prefixes) <= 1
    idx = bisect_right(kept, w)
    pred = kept[idx - 1] if idx else None
    if prefixes:
        assert prefixes[0] == pred
    checked += 1
print("đã kiểm", checked, "trường hợp: mệnh đề người tiền nhiệm đúng")

đã kiểm 2000 trường hợp: mệnh đề người tiền nhiệm đúng


## 11. Time / Space Complexity

Gọi `n` = số root, `m` = độ dài root tối đa (`<= 100`), `W` = số từ, `L` = tổng độ dài các từ trong câu, `S` = tổng độ dài các root.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn `startswith` | `O(W * n * m)` | `O(L)` |
| 2. Tập băm tiền tố | `O(W * m^2)` | `O(S)` |
| 3. Trie (nộp) | `O(S + L)` (thực tế `O(S + W * m)`) | `O(S)` |
| 4. Sắp xếp + `bisect` | `O(n m log n + W m log n)` | `O(S)` |

Với Trie: mỗi từ đi tối đa `min(|w|, m)` bước, **không phụ thuộc `n`**. Ta đếm số bước đi thật và số nút.

In [23]:
def count_trie_steps(dictionary, sentence):
    trie = Trie()
    nodes = 0
    for r in dictionary:
        node = trie
        for ch in r:
            if ch not in node.children:
                node.children[ch] = Trie()
                nodes += 1
            node = node.children[ch]
        node.is_end = True
    steps = 0
    for w in sentence.split():
        node = trie
        for ch in w:
            steps += 1
            if ch not in node.children:
                break
            node = node.children[ch]
            if node.is_end:
                break
    return nodes, steps

rng = random.Random(11)
for n_roots, root_len, n_words, word_len in [(10, 5, 20, 20), (1000, 100, 1000, 990)]:
    dictionary = ["".join(rng.choice("ab") for _ in range(root_len)) for _ in range(n_roots)]
    sentence = " ".join("".join(rng.choice("ab") for _ in range(word_len)) for _ in range(n_words))
    nodes, steps = count_trie_steps(dictionary, sentence)
    print(f"roots={n_roots} (độ dài {root_len}), từ={n_words} (độ dài {word_len}): nút = {nodes}, bước đi = {steps}, "
          f"độ dài câu = {len(sentence)}, W*m = {n_words * root_len}")
    assert steps <= n_words * root_len

roots=10 (độ dài 5), từ=20 (độ dài 20): nút = 27, bước đi = 75, độ dài câu = 419, W*m = 100


roots=1000 (độ dài 100), từ=1000 (độ dài 990): nút = 91163, bước đi = 11337, độ dài câu = 990999, W*m = 100000


## 12. Bộ test

Gồm: ví dụ đề, biên (một root một từ, từ ngắn hơn root, từ bằng root, root lồng nhau, root trùng nhau, không có từ nào được thay), phản ví dụ cho lời giải sai, đối chiếu 4 cách bằng `assert` trên nhiều input (có vét cạn trên bảng chữ cái nhỏ), test ngẫu nhiên có seed đúng ràng buộc, và test ở kích thước lớn nhất.

In [24]:
ALL_METHODS = {
    "brute": replace_brute,
    "prefix_set": replace_prefix_set,
    "trie": replace_trie,
    "bisect": replace_bisect,
    "leetcode": Solution().replaceWords,
}

fixed_cases = [
    (["cat", "bat", "rat"], "the cattle was rattled by the battery", "the cat was rat by the bat"),
    (["a", "b", "c"], "aadsfasf absbs bbab cadsfafs", "a a b c"),
    (["a"], "a", "a"),                                           # một root một từ, bằng nhau
    (["hello"], "hell", "hell"),                                 # từ ngắn hơn root: giữ nguyên
    (["hello"], "hello", "hello"),                               # từ bằng root
    (["hello"], "helloworld", "hello"),
    (["cat", "c"], "cattle", "c"),                               # lồng nhau: chọn ngắn nhất
    (["c", "cat"], "cattle", "c"),                               # thứ tự trong từ điển không ảnh hưởng
    (["cat", "cat"], "cat cats", "cat cat"),                     # root trùng
    (["x", "y"], "abc def", "abc def"),                          # không từ nào được thay
    (["ab", "abc", "abcd"], "abcde abcd abc ab a", "ab ab ab ab a"),
    (["catt", "cat", "bat", "rat"], "the cattle was rattled by the battery", "the cat was rat by the bat"),
    (["e", "k", "c", "harqp", "h", "gsafc", "vn", "lqp", "soy", "mr", "x", "iitgm", "sb", "oo", "spj", "gwmly", "iu", "z", "f", "ha", "vds", "v", "vpx", "fir", "t", "xo", "apifm", "tlznm", "kkv", "nxyud", "j", "qp", "omn", "zoxp", "mutu", "i", "nxth", "dwuer", "sadl", "pv", "w", "mdbn", "qmxlr", "i", "pwwj"],
     "ikkbp miszkays wqjferqoxjwvbieyk gvcfldkiavww vhokchxz dvypwyb bxahfzcfanteibiltins ueebf lqhflvwxksi dco kddxmckhvqifbuzkhstp wc ytzzlm gximjuhzfdjuamhsu gdkbmhpnvy ifvifdb msqbllwpjnglnhfxvbjzjkmhnzhdnhjjxv ofwfejbtdjfpkgcksvsv w bjcpnaltsjdpbjdlu ukoalpwxzstdnb"
     , None),
]
for dictionary, sentence, expected in fixed_cases:
    results = {name: fn(list(dictionary), sentence) for name, fn in ALL_METHODS.items()}
    ref = results["brute"]
    for name, got in results.items():
        assert got == ref, (name, dictionary, sentence, got, ref)
    if expected is not None:
        assert ref == expected, (dictionary, sentence, ref, expected)
print("test cố định:", len(fixed_cases), "bộ x", len(ALL_METHODS), "cách: OK")

test cố định: 13 bộ x 5 cách: OK


In [25]:
# Phản ví dụ 1: lấy root khớp ĐẦU TIÊN trong từ điển (thay vì NGẮN nhất)
def wrong_first_match(dictionary, sentence):
    out = []
    for w in sentence.split():
        for r in dictionary:
            if w.startswith(r):
                w = r
                break
        out.append(w)
    return " ".join(out)

print(wrong_first_match(["cat", "c"], "cattle"), "| đúng:", replace_trie(["cat", "c"], "cattle"))

# Phản ví dụ 2: dùng "root in word" thay vì kiểm tra TIỀN TỐ
def wrong_substring(dictionary, sentence):
    out = []
    for w in sentence.split():
        best = w
        for r in dictionary:
            if r in w and len(r) < len(best):
                best = r
        out.append(best)
    return " ".join(out)

print(wrong_substring(["cat"], "the concatenate"), "| đúng:", replace_trie(["cat"], "the concatenate"))

cat | đúng: c
the cat | đúng: the concatenate


In [26]:
# Vét cạn trên bảng chữ cái {a, b}: mọi từ điển nhỏ x mọi câu nhỏ, so brute với các cách còn lại
from itertools import product

words_pool = ["".join(t) for n in range(1, 5) for t in product("ab", repeat=n)]     # 30 từ
rng = random.Random(12)
cases = 0
for _ in range(4000):
    dictionary = rng.sample(words_pool, rng.randint(1, 5))
    sentence = " ".join(rng.choice(words_pool) for _ in range(rng.randint(1, 6)))
    ref = replace_brute(dictionary, sentence)
    for name in ["prefix_set", "trie", "bisect", "leetcode"]:
        assert ALL_METHODS[name](dictionary, sentence) == ref, (name, dictionary, sentence)
    cases += 1
print("ngẫu nhiên có cấu trúc:", cases, "trường hợp: các cách đều khớp")

# Vét cạn đầy đủ: mọi từ điển gồm đúng 2 root và mọi từ độ dài <= 4 trên {a, b}
cnt = 0
for r1 in words_pool:
    for r2 in words_pool:
        d = [r1, r2]
        for w in words_pool:
            ref = replace_brute(d, w)
            assert replace_trie(d, w) == ref == replace_bisect(d, w) == replace_prefix_set(d, w), (d, w)
            cnt += 1
print("vét cạn 2 root x mọi từ:", cnt, "trường hợp OK")

ngẫu nhiên có cấu trúc: 4000 trường hợp: các cách đều khớp


vét cạn 2 root x mọi từ: 27000 trường hợp OK


In [27]:
# Ngẫu nhiên có seed, đúng ràng buộc
rng = random.Random(648)

def random_case(rng, alphabet, max_roots, max_root_len, max_words, max_word_len):
    dictionary = ["".join(rng.choice(alphabet) for _ in range(rng.randint(1, max_root_len)))
                  for _ in range(rng.randint(1, max_roots))]
    sentence = " ".join("".join(rng.choice(alphabet) for _ in range(rng.randint(1, max_word_len)))
                        for _ in range(rng.randint(1, max_words)))
    return dictionary, sentence

for _ in range(500):
    alphabet = rng.choice(["a", "ab", "abc", "abcdefghijklmnopqrstuvwxyz"])
    dictionary, sentence = random_case(rng, alphabet, 8, 6, 10, 12)
    ref = replace_brute(dictionary, sentence)
    for name, fn in ALL_METHODS.items():
        assert fn(dictionary, sentence) == ref, (name, dictionary, sentence)
print("ngẫu nhiên nhỏ: OK")

# Kích thước lớn nhất: 1000 root (độ dài <= 100), 1000 từ (độ dài <= 990), tổng câu <= 10^6
for alphabet in ["ab", "abcdefghijklmnopqrstuvwxyz"]:
    dictionary = ["".join(rng.choice(alphabet) for _ in range(rng.randint(1, 100))) for _ in range(1000)]
    words = []
    for _ in range(1000):
        base = rng.choice(dictionary) if rng.random() < 0.6 else ""         # nhiều từ thật sự bắt đầu bằng một root
        tail = "".join(rng.choice(alphabet) for _ in range(rng.randint(1, 990 - len(base))))
        words.append(base + tail)
    sentence = " ".join(words)
    assert len(sentence) <= 10 ** 6 and len(words) <= 1000
    ref = replace_brute(dictionary, sentence)
    for name in ["prefix_set", "trie", "bisect", "leetcode"]:
        assert ALL_METHODS[name](dictionary, sentence) == ref, name
    replaced = sum(1 for a, b in zip(words, ref.split()) if a != b)
    print(f"alphabet {len(alphabet)} chữ: câu dài {len(sentence)}, số từ bị thay = {replaced}: OK")

ngẫu nhiên nhỏ: OK


alphabet 2 chữ: câu dài 504805, số từ bị thay = 1000: OK


alphabet 26 chữ: câu dài 509334, số từ bị thay = 694: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Lấy root khớp **đầu tiên** trong từ điển, không phải **ngắn nhất**

Đề yêu cầu root **ngắn nhất**. Nếu từ điển là `["cat", "c"]` và từ là `"cattle"`, vòng lặp gặp `"cat"` trước và dừng, trả `cat` thay vì `c` (đã thấy ở mục 12). Sửa: duyệt mọi root và lấy ngắn nhất, hoặc dùng Trie/duyệt tiền tố ngắn → dài.

### Lỗi 2 — Trie đi tiếp quá `is_end` (lấy root dài nhất)

Nếu bỏ lệnh `return` ngay khi gặp `is_end`, ta tiếp tục xuống sâu và ghi đè bằng root dài hơn.

In [28]:
def shortest_root_wrong_longest(trie, word):
    node = trie
    best = word
    for k, ch in enumerate(word):
        if ch not in node.children:
            break
        node = node.children[ch]
        if node.is_end:
            best = word[:k + 1]      # SAI: không dừng, tiếp tục và ghi đè bằng root dài hơn
    return best

t = Trie()
for r in ["c", "cat", "cattle"]:
    t.insert(r)
print("sai  :", shortest_root_wrong_longest(t, "cattleship"))
print("đúng :", t.shortest_root("cattleship"))

sai  : cattle
đúng : c


### Lỗi 3 — Dùng `root in word` (chuỗi con) thay vì tiền tố

Đã thấy ở mục 12: `wrong_substring(["cat"], "the concatenate")` thay `concatenate` bằng `cat` vì `cat` nằm **giữa** từ. Root phải ở **đầu** từ: dùng `startswith` (hoặc đi theo Trie từ gốc).

### Lỗi 4 — Trả về tiền tố "đi dở" khi chưa gặp root

Khi đường trong Trie đứt hoặc hết từ mà chưa gặp `is_end`, phải trả **nguyên từ gốc**. Nhiều bạn trả `word[:k]` (phần đã đi được) — sai vì phần đó chưa phải root.

In [29]:
def shortest_root_wrong_partial(trie, word):
    node = trie
    k = 0
    for k, ch in enumerate(word):
        if ch not in node.children:
            return word[:k]          # SAI: trả phần đã khớp dù nó không phải root
        node = node.children[ch]
        if node.is_end:
            return word[:k + 1]
    return word

t = Trie()
t.insert("cat")
for w in ["cattle", "cab", "cx"]:
    print(f"{w!r}: sai -> {shortest_root_wrong_partial(t, w)!r}, đúng -> {t.shortest_root(w)!r}")

'cattle': sai -> 'cat', đúng -> 'cat'
'cab': sai -> 'ca', đúng -> 'cab'
'cx': sai -> 'c', đúng -> 'cx'


Ở `"cab"`: đường `c → a` còn đi được nhưng `b` không có, `"ca"` **không** là root (root duy nhất là `"cat"`), nên phải giữ nguyên `"cab"`.

### Lỗi 5 — Quên rằng root có thể **trùng nhau** hoặc **lồng nhau** khi xây cấu trúc

Khi xây `set` hay Trie từ `dictionary`, root trùng không gây hại (`set` tự loại, Trie ghi `is_end = True` lần nữa). Nhưng nếu bạn tự sắp xếp theo độ dài hoặc xóa root "dài" mà quên trường hợp trùng, dễ lệch. Test luôn có `["cat", "cat"]` và `["a", "ab", "abc"]`.

In [30]:
for d, s in [(["cat", "cat"], "cat cats"), (["a", "ab", "abc"], "abc abd b")]:
    print(d, s, "->", Solution().replaceWords(d, s))

['cat', 'cat'] cat cats -> cat cat
['a', 'ab', 'abc'] abc abd b -> a a b


## 14. Mẹo & nhận dạng pattern

- **"Tiền tố" xuất hiện trong đề** (prefix, root, autocomplete, từ điển) → nghĩ ngay tới **Trie**.
- Trie rất hợp để trả lời "**tiền tố ngắn nhất / dài nhất** của chuỗi có trong từ điển": đi dọc từ gốc, dừng ở `is_end` đầu tiên (ngắn nhất) hoặc ghi nhận `is_end` cuối cùng (dài nhất).
- Khi ràng buộc nhỏ, **`set` + thử từng tiền tố** thường đủ nhanh và ngắn hơn Trie; cân nhắc `m` (độ dài root tối đa) để chặn vòng thử.
- `dict` lồng dict, hoặc class `children = {}` với `is_end`, đều cài được Trie; mảng 26 phần tử nhanh hơn một chút nhưng dài hơn.
- Họ bài liên quan: *Implement Trie* (208), *Design Add and Search Words* (211), *Map Sum Pairs* (677), *Short Encoding of Words* (820), *Longest Word in Dictionary* (720).
- Nhớ `split()`/`join()`: `" ".join(...)` để ghép lại câu, đừng nối bằng `+=` rồi quên bỏ dấu cách cuối.

## 15. Biến thể

### 15.1. Thay bằng root **dài nhất**

Đảo yêu cầu: trong Trie, đi tiếp tới khi đường đứt, ghi nhớ root hoàn chỉnh gần nhất.

### 15.2. Đếm số từ bị thay

Đếm các từ mà kết quả khác từ gốc.

### 15.3. Root là **hậu tố** (suffix)

Đảo ngược cả root lẫn từ rồi dùng lại thuật toán; đảo ngược kết quả. (Hậu tố = đoạn ở cuối từ.)

In [31]:
def replace_longest(dictionary, sentence):
    """15.1: thay bằng root DÀI NHẤT là tiền tố của từ."""
    trie = Trie()
    for r in dictionary:
        trie.insert(r)
    out = []
    for w in sentence.split():
        node, best = trie, w
        for k, ch in enumerate(w):
            if ch not in node.children:
                break
            node = node.children[ch]
            if node.is_end:
                best = w[:k + 1]
        out.append(best)
    return " ".join(out)

def count_replaced(dictionary, sentence):
    """15.2: số từ bị thay đổi."""
    new = replace_trie(dictionary, sentence).split()
    return sum(1 for a, b in zip(sentence.split(), new) if a != b)

def replace_suffix(dictionary, sentence):
    """15.3: root là hậu tố; thay bằng hậu tố NGẮN nhất."""
    rev_dict = [r[::-1] for r in dictionary]
    rev_words = [w[::-1] for w in sentence.split()]
    return " ".join(x[::-1] for x in replace_trie(rev_dict, " ".join(rev_words)).split())

print(replace_longest(["c", "cat", "cattle"], "cattleship catalog"))
print(count_replaced(["cat", "bat", "rat"], "the cattle was rattled by the battery"))
print(replace_suffix(["ing", "ed"], "walking walked talk"))

# kiểm chứng bằng oracle độc lập
def longest_oracle(dictionary, sentence):
    out = []
    for w in sentence.split():
        best = w
        found = [r for r in dictionary if w.startswith(r)]
        if found:
            best = max(found, key=len)
        out.append(best)
    return " ".join(out)

def suffix_oracle(dictionary, sentence):
    out = []
    for w in sentence.split():
        found = [r for r in dictionary if w.endswith(r)]
        out.append(w[len(w) - len(min(found, key=len)):] if found else w)
    return " ".join(out)

rng = random.Random(15)
for _ in range(500):
    d, s = random_case(rng, "ab", 6, 4, 6, 8)
    assert replace_longest(d, s) == longest_oracle(d, s)
    assert replace_suffix(d, s) == suffix_oracle(d, s)
print("các biến thể 15.1-15.3 OK")

cattle cat
3
ing ed talk
các biến thể 15.1-15.3 OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn `startswith` | Mỗi từ thử mọi root | `O(W n m)` | `O(L)` | Chỉ để hiểu đề / oracle |
| 2. Tập băm tiền tố | Thử tiền tố ngắn → dài trong `set` | `O(W m^2)` | `O(S)` | Ràng buộc nhỏ, viết nhanh |
| 3. Trie | Đi dọc từ, dừng ở `is_end` đầu tiên | `O(S + L)` | `O(S)` | **Nộp bài**, tổng quát, nhiều truy vấn |
| 4. Sắp xếp + `bisect` | Root tối giản; root là người tiền nhiệm | `O((n+W) m log n)` | `O(S)` | Mở rộng tư duy; không cần cấu trúc cây |

In [32]:
rng = random.Random(16)
dictionary = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 30))) for _ in range(1000)]
sentence = " ".join("".join(rng.choice("abc") for _ in range(rng.randint(1, 500))) for _ in range(1000))
answers = {name: fn(dictionary, sentence) for name, fn in ALL_METHODS.items()}
print({name: len(v) for name, v in answers.items()})
assert len({v for v in answers.values()}) == 1

{'brute': 1999, 'prefix_set': 1999, 'trie': 1999, 'bisect': 1999, 'leetcode': 1999}


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của bốn dòng sau, rồi chạy để kiểm tra:

```python
print("the cat sat".split())
print("cattle".startswith("cat"), "cattle".startswith("tle"))
print(" ".join(["a", "b", "c"]))
print("cattle"[:3], "ab" in "cab", "cab".startswith("ab"))
```

In [33]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None
my_guess_line4 = None

### Bài 2 — Tính tay Trie (dễ-vừa)

Với `dictionary = ["a", "ab", "bc"]` và câu `"abcd bcd xa"`: (a) trie có bao nhiêu nút (không tính gốc)? (b) câu sau khi thay là gì?

In [34]:
# Bài 2: điền kết quả tính tay
hand_nodes = None       # số nút của trie (không tính gốc)
hand_result = None      # câu sau khi thay

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành `insert` và `shortest_root` của Trie.

In [35]:
class ExerciseTrie:
    def __init__(self):
        self.children = {}
        self.is_end = False

    def insert(self, word):
        node = self
        # TODO 1: với mỗi ký tự ch của word: nếu ch chưa có trong node.children thì tạo ExerciseTrie() mới
        # TODO 2: đi xuống nút con
        # TODO 3: sau vòng lặp, đánh dấu node.is_end = True
        pass

    def shortest_root(self, word):
        node = self
        # TODO 4: với mỗi (k, ch): nếu ch không có -> return word
        # TODO 5: đi xuống; nếu nút có is_end -> return word[:k + 1]
        # TODO 6: hết vòng lặp -> return word
        return word


# t = ExerciseTrie()
# for r in ["cat", "bat", "rat"]:
#     t.insert(r)
# assert t.shortest_root("cattle") == "cat"
# assert t.shortest_root("by") == "by"
# assert t.shortest_root("the") == "the"

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 2, hãy tự viết `exercise_4_replace(dictionary, sentence)` bằng `set` các root và thử từng tiền tố (nhớ giới hạn `k` theo độ dài root tối đa).

In [36]:
def exercise_4_replace(dictionary: List[str], sentence: str) -> str:
    # TODO
    pass


# assert exercise_4_replace(["cat", "bat", "rat"], "the cattle was rattled by the battery") == "the cat was rat by the bat"
# assert exercise_4_replace(["a", "b", "c"], "aadsfasf absbs bbab cadsfafs") == "a a b c"
# assert exercise_4_replace(["cat", "c"], "cattle") == "c"

### Bài 5 — Mini challenge: thay bằng root **dài nhất** (khó)

Giống đề gốc nhưng mỗi từ được thay bằng **root DÀI NHẤT** là tiền tố của nó (từ không có tiền tố là root thì giữ nguyên). Viết `exercise_5_longest(dictionary, sentence)`.

```python
exercise_5_longest(["c", "cat", "cattle"], "cattleship catalog dog") -> "cattle cat dog"
exercise_5_longest(["a", "ab"], "abc a b")                            -> "ab a b"
```

In [37]:
def exercise_5_longest(dictionary: List[str], sentence: str) -> str:
    # TODO
    pass


# assert exercise_5_longest(["c", "cat", "cattle"], "cattleship catalog dog") == "cattle cat dog"
# assert exercise_5_longest(["a", "ab"], "abc a b") == "ab a b"
# assert exercise_5_longest(["x"], "y z") == "y z"

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `split()` cắt theo khoảng trắng; `startswith` chỉ nhìn đầu chuỗi.
- Mức 2: `"ab" in "cab"` là `True` (chuỗi con) nhưng `"cab".startswith("ab")` là `False`.
- Mức 3: `['the', 'cat', 'sat']`; `True False`; `a b c`; `cat True False`.

**Bài 2**
- Mức 1: vẽ trie: nhánh `a` (có `is_end`), con `b` (`is_end`); nhánh `b` có con `c` (`is_end`).
- Mức 2: nút: `a`, `a→b`, `b`, `b→c` = 4 nút.
- Mức 3: `abcd` → gặp `a` (is_end) ngay → `a`; `bcd` → `b` (chưa end), `c` (end) → `bc`; `xa` → `x` không có → `xa`. Câu: `"a bc xa"`.

**Bài 3**
- Mức 1: giống Trie ở mục 7.2.
- Mức 2: `if ch not in node.children: node.children[ch] = ExerciseTrie()`.
- Mức 3: `shortest_root`: `for k, ch in enumerate(word)`; đứt → `return word`; sau khi xuống nút, `if node.is_end: return word[:k + 1]`.

**Bài 4**
- Mức 1: `roots = set(dictionary)`, `max_len = max(map(len, roots))`.
- Mức 2: với mỗi từ, `for k in range(1, min(len(w), max_len) + 1): if w[:k] in roots: ...`.
- Mức 3: gặp tiền tố đầu tiên thuộc `roots` thì thay và `break`; cuối cùng `" ".join(...)`.

**Bài 5**
- Mức 1: dùng Trie như cũ nhưng **không dừng** khi gặp `is_end`.
- Mức 2: lưu `best = w[:k + 1]` mỗi khi gặp `is_end`, tiếp tục đi tới khi đường đứt hoặc hết từ.
- Mức 3: khởi tạo `best = w`; trả `best` sau vòng lặp. (Hoặc với `set`: thử `k` **giảm dần** từ `min(len(w), max_len)` tới `1`.)

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [38]:
print("the cat sat".split())
print("cattle".startswith("cat"), "cattle".startswith("tle"))
print(" ".join(["a", "b", "c"]))
print("cattle"[:3], "ab" in "cab", "cab".startswith("ab"))

assert "the cat sat".split() == ["the", "cat", "sat"]
assert ("cattle".startswith("cat"), "cattle".startswith("tle")) == (True, False)
assert " ".join(["a", "b", "c"]) == "a b c"
assert ("cattle"[:3], "ab" in "cab", "cab".startswith("ab")) == ("cat", True, False)

['the', 'cat', 'sat']
True False
a b c
cat True False


### Đáp án Bài 2

In [39]:
nodes, _ = count_trie_steps(["a", "ab", "bc"], "abcd bcd xa")
res = replace_trie(["a", "ab", "bc"], "abcd bcd xa")
print("số nút:", nodes, "| kết quả:", repr(res))
assert nodes == 4
assert res == "a bc xa"

số nút: 4 | kết quả: 'a bc xa'


### Đáp án Bài 3

In [40]:
class ExerciseTrieSolution:
    def __init__(self):
        self.children = {}
        self.is_end = False

    def insert(self, word):
        node = self
        for ch in word:
            if ch not in node.children:
                node.children[ch] = ExerciseTrieSolution()
            node = node.children[ch]
        node.is_end = True

    def shortest_root(self, word):
        node = self
        for k, ch in enumerate(word):
            if ch not in node.children:
                return word
            node = node.children[ch]
            if node.is_end:
                return word[:k + 1]
        return word

t = ExerciseTrieSolution()
for r in ["cat", "bat", "rat"]:
    t.insert(r)
assert t.shortest_root("cattle") == "cat"
assert t.shortest_root("by") == "by"
assert t.shortest_root("the") == "the"
assert t.shortest_root("ca") == "ca"
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [41]:
def exercise_4_solution(dictionary: List[str], sentence: str) -> str:
    roots = set(dictionary)
    max_len = max(map(len, roots))
    out = []
    for w in sentence.split():
        for k in range(1, min(len(w), max_len) + 1):
            if w[:k] in roots:
                w = w[:k]
                break
        out.append(w)
    return " ".join(out)

assert exercise_4_solution(["cat", "bat", "rat"], "the cattle was rattled by the battery") == "the cat was rat by the bat"
assert exercise_4_solution(["a", "b", "c"], "aadsfasf absbs bbab cadsfafs") == "a a b c"
assert exercise_4_solution(["cat", "c"], "cattle") == "c"
rng = random.Random(44)
for _ in range(300):
    d, s = random_case(rng, "ab", 6, 4, 6, 8)
    assert exercise_4_solution(d, s) == replace_brute(d, s)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [42]:
def exercise_5_solution(dictionary: List[str], sentence: str) -> str:
    roots = set(dictionary)
    max_len = max(map(len, roots))
    out = []
    for w in sentence.split():
        for k in range(min(len(w), max_len), 0, -1):       # từ dài xuống ngắn
            if w[:k] in roots:
                w = w[:k]
                break
        out.append(w)
    return " ".join(out)

assert exercise_5_solution(["c", "cat", "cattle"], "cattleship catalog dog") == "cattle cat dog"
assert exercise_5_solution(["a", "ab"], "abc a b") == "ab a b"
assert exercise_5_solution(["x"], "y z") == "y z"
rng = random.Random(55)
for _ in range(300):
    d, s = random_case(rng, "ab", 6, 4, 6, 8)
    assert exercise_5_solution(d, s) == longest_oracle(d, s) == replace_longest(d, s)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template Trie

```python
class Trie:
    def __init__(self):
        self.children = {}
        self.is_end = False

    def insert(self, word):
        node = self
        for ch in word:
            if ch not in node.children:
                node.children[ch] = Trie()
            node = node.children[ch]
        node.is_end = True

    def shortest_root(self, word):
        node = self
        for k, ch in enumerate(word):
            if ch not in node.children:
                return word              # đường đứt
            node = node.children[ch]
            if node.is_end:
                return word[:k + 1]      # gặp root đầu tiên = ngắn nhất
        return word                      # hết từ, chưa gặp root


class Solution:
    def replaceWords(self, dictionary: List[str], sentence: str) -> str:
        trie = Trie()
        for root in dictionary:
            trie.insert(root)
        return " ".join(trie.shortest_root(w) for w in sentence.split())
```

Câu thần chú: **"đi dọc từ trong Trie; gặp `is_end` đầu tiên thì cắt ngay; đứt đường hoặc hết từ thì giữ nguyên."**

### Checklist trước khi submit

- [ ] Root phải là **tiền tố** (đầu từ), không phải chuỗi con (`startswith`/đi từ gốc Trie, không dùng `in`).
- [ ] Chọn root **ngắn nhất**: dừng ngay ở `is_end` đầu tiên.
- [ ] Không gặp root thì trả **nguyên từ gốc** (không trả tiền tố đi dở).
- [ ] Root trùng / lồng nhau / dài hơn từ đều xử lý đúng (`["cat", "c"]`, `["hello"]` với `"hell"`).
- [ ] `split()` rồi `" ".join(...)` để giữ đúng định dạng câu.
- [ ] Skeleton đúng: `class Solution`, method `replaceWords(self, dictionary: List[str], sentence: str) -> str`; lớp phụ `Trie` đặt phía trên `Solution`.